# Metrics, Thresholds & Imbalanced Data
**Topics:** Confusion Matrix · Precision/Recall/F1 · ROC-AUC vs PR-AUC · Threshold Selection ·
Calibration · Class Imbalance · Offline↔Online Metric Alignment

> 📌 **If you're transitioning from a DS role, this is the chapter to read carefully.** You
> almost certainly know precision and recall. What's less commonly carried over from analytics
> work is: choosing an operating threshold against an explicit business cost, treating
> calibration as separate from ranking quality, and connecting an offline metric to an online
> one you can actually move. Those three things are most of what "productionizing a model"
> means in practice.

## 1. The Confusion Matrix and What Comes From It

|  | Predicted positive | Predicted negative |
|---|---|---|
| **Actually positive** | TP | FN |
| **Actually negative** | FP | TN |

| Metric | Formula | Reads as |
|---|---|---|
| Precision | TP / (TP + FP) | Of the things I flagged, how many were right? |
| Recall (TPR) | TP / (TP + FN) | Of the things I should have flagged, how many did I catch? |
| Specificity (TNR) | TN / (TN + FP) | Of the negatives, how many did I correctly leave alone? |
| F1 | Harmonic mean of P and R | One number when you have no cost information |
| FPR | FP / (FP + TN) | What fraction of negatives did I wrongly flag? |

**F1 is a fallback, not a goal.** It weights precision and recall equally, which is almost never
what the business wants. Use it to compare models when you genuinely have no cost information;
replace it with an explicit cost function as soon as you do.

### Accuracy is a trap under imbalance
At 0.5% fraud, a model that predicts "never fraud" scores 99.5% accuracy and has zero value.
This is the single most common interview trap in this area.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, accuracy_score)

rng = np.random.default_rng(0)

# 2% positive rate — a realistic fraud / churn / defect setting.
X, y = make_classification(n_samples=20000, n_features=20, n_informative=6,
                            weights=[0.98, 0.02], flip_y=0.01, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

model = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
proba = model.predict_proba(Xte)[:, 1]

print(f"Positive rate in test set: {yte.mean():.2%}\n")
print(f"{'model':<28}{'accuracy':>10}{'precision':>11}{'recall':>9}{'F1':>8}")
print("-" * 66)

trivial = np.zeros_like(yte)
print(f"{'always predict negative':<28}{accuracy_score(yte, trivial):>10.4f}"
      f"{0.0:>11.4f}{0.0:>9.4f}{0.0:>8.4f}")

pred = (proba >= 0.5).astype(int)
print(f"{'logistic @ threshold 0.5':<28}{accuracy_score(yte, pred):>10.4f}"
      f"{precision_score(yte, pred, zero_division=0):>11.4f}"
      f"{recall_score(yte, pred):>9.4f}{f1_score(yte, pred):>8.4f}")

print()
print("The useless model beats nothing but scores 97.9% accuracy.")
print("Report accuracy on an imbalanced problem and you have said almost nothing.")

## 2. ROC-AUC vs PR-AUC

Both summarize performance across *all* thresholds. They disagree, and the disagreement matters.

| | ROC-AUC | PR-AUC (average precision) |
|---|---|---|
| Axes | TPR vs **FPR** | Precision vs Recall |
| Baseline (random) | 0.5 always | The positive rate |
| Under heavy imbalance | Stays optimistic | Drops honestly |
| Answers | "Does it rank a random positive above a random negative?" | "If I act on the top predictions, how many are right?" |

**Why ROC-AUC misleads under imbalance.** FPR has TN in the denominator. When negatives vastly
outnumber positives, thousands of false positives barely move FPR — so the ROC curve stays
flattering while precision collapses. PR-AUC has no TN term anywhere, so it reflects what a
person acting on the alerts would actually experience.

**Rule:** when the positive class is rare *and* you'll act on individual predictions, lead with
PR-AUC. ROC-AUC is fine for balanced problems and for comparing rankers.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

def compare(pos_rate, n=40000, seed=1):
    X, y = make_classification(n_samples=n, n_features=20, n_informative=6,
                                weights=[1 - pos_rate, pos_rate], flip_y=0.01,
                                random_state=seed)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    p = LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, p), average_precision_score(yte, p), yte.mean()

print(f"{'positive rate':>14}{'ROC-AUC':>10}{'PR-AUC':>9}{'PR baseline':>13}{'ROC baseline':>14}")
print("-" * 62)
for rate in [0.50, 0.20, 0.05, 0.01, 0.002]:
    roc, pr, actual = compare(rate)
    print(f"{actual:>13.2%}{roc:>10.3f}{pr:>9.3f}{actual:>13.3f}{0.5:>14.1f}")

print()
print("ROC-AUC barely moves as the problem gets 250x harder.")
print("PR-AUC tracks the difficulty — and its baseline moves with it, so a PR-AUC of")
print("0.30 at a 0.2% positive rate is a strong model, while 0.30 at 50% is a broken one.")
print("Always report the baseline alongside PR-AUC; the number alone is uninterpretable.")

## 3. Threshold Selection: Where the Business Enters

A classifier outputs a score. **The threshold is a separate decision, and it is not a modeling
decision** — it's a business one. Training a better model and choosing a better threshold are
independent levers, and the second is usually faster and cheaper.

`0.5` is a default, not an answer. It is only optimal when classes are balanced and the two
error types cost the same — which is almost never true.

### How to actually choose one

1. **Write down the cost of each error.** A missed fraud costs the chargeback; a false positive
   costs a blocked customer and a support ticket. Get real numbers from whoever owns the P&L.
2. **Minimize expected cost across thresholds.** This is a one-line sweep.
3. **Or optimize under a constraint** — "maximize recall subject to precision ≥ 0.80," or "the
   review team can handle 500 cases a day." Capacity constraints are extremely common and often
   decide the threshold outright.
4. **Re-check it when the base rate shifts.** A threshold tuned at 2% fraud is wrong at 6%.

> 💡 **Interview Tip:** Asked "how would you pick the threshold," the weak answer is "tune F1."
> The strong answer asks a question back: *"What does a false positive cost relative to a false
> negative, and is there a capacity limit on the downstream action?"* That single question
> signals more production experience than any metric you could name.

In [ ]:
def expected_cost(y_true, scores, thr, cost_fp, cost_fn):
    pred = (scores >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return fp * cost_fp + fn * cost_fn, fp, fn, tp

thresholds = np.linspace(0.01, 0.99, 99)

print("Same model and scores. Only the cost ratio changes.\n")
print(f"{'scenario':<34}{'best thr':>10}{'precision':>11}{'recall':>9}{'flagged':>9}")
print("-" * 73)

scenarios = [
    ("FN 100x worse (fraud loss)",      1,   100),
    ("FN 10x worse",                     1,    10),
    ("symmetric costs",                  1,     1),
    ("FP 5x worse (blocking a customer)", 5,    1),
]

for name, c_fp, c_fn in scenarios:
    costs = [expected_cost(yte, proba, t, c_fp, c_fn)[0] for t in thresholds]
    best = thresholds[int(np.argmin(costs))]
    pred = (proba >= best).astype(int)
    print(f"{name:<34}{best:>10.2f}"
          f"{precision_score(yte, pred, zero_division=0):>11.3f}"
          f"{recall_score(yte, pred):>9.3f}{pred.sum():>9}")

print()
# Capacity-constrained variant — often the real-world binding constraint
CAPACITY = 100
k = np.argsort(-proba)[:CAPACITY]
caught = yte[k].sum()
print(f"Capacity constraint instead of cost: review team handles {CAPACITY} cases/day.")
print(f"  Threshold = the {CAPACITY}th highest score = {np.sort(proba)[-CAPACITY]:.3f}")
print(f"  Precision@{CAPACITY} = {caught / CAPACITY:.3f}   "
      f"recall@{CAPACITY} = {caught / yte.sum():.3f}")
lo = thresholds[int(np.argmin([expected_cost(yte, proba, t, 1, 100)[0] for t in thresholds]))]
hi = thresholds[int(np.argmin([expected_cost(yte, proba, t, 5, 1)[0] for t in thresholds]))]
print()
print(f"The threshold moved from {lo:.2f} to {hi:.2f} without retraining anything,")
print("and recall ranged from 0.80 down to 0.20 across those settings.")
print("That range is why 'we improved AUC by 0.01' is often a much smaller win")
print("than 'we set the threshold correctly for the first time'.")

## 4. Calibration

**Ranking quality and probability quality are different properties.** A model can rank perfectly
(AUC 1.0) while every predicted probability is wrong. Squaring all scores preserves order
exactly and destroys calibration.

A model is **calibrated** if, among predictions of 0.3, roughly 30% are actually positive.

### When calibration matters
- **Anything that multiplies the probability by a value.** Expected-value decisions,
  ad auctions (bid = pCTR × value), expected loss, inventory. Miscalibration corrupts the
  arithmetic directly.
- **Thresholds with a business meaning.** "Escalate above 80% confidence" is meaningless if 80%
  actually means 40%.
- **Showing scores to humans**, who will interpret them as probabilities whether or not you
  intend it.

### When it doesn't
Pure ranking with a fixed cut — top-k retrieval, sorting a feed. Order is all that's used.

### What breaks calibration
- **Class rebalancing.** Oversampling, undersampling, and `class_weight='balanced'` all shift
  the base rate the model sees, so its outputs no longer reflect true probabilities. This is the
  most common cause in practice and it surprises people.
- Regularization, some tree ensembles (which tend to be over-confident near 0 and 1), and SVMs
  (whose decision function isn't a probability at all).

### Fixes
**Platt scaling** (fit a logistic regression on the scores) and **isotonic regression** (fit a
non-decreasing step function; more flexible, needs more data). Both must be fitted on a
*held-out* set — calibrating on the training set achieves nothing.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss

def ece(y_true, prob, bins=10):
    """Expected Calibration Error: mean |confidence - accuracy| weighted by bin size."""
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (prob > lo) & (prob <= hi)
        if m.sum():
            total += m.mean() * abs(prob[m].mean() - y_true[m].mean())
    return total

Xcal, Xho, ycal, yho = train_test_split(Xte, yte, test_size=0.5, stratify=yte, random_state=0)

base = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
# The rebalanced model: better recall at 0.5, but its probabilities now lie.
weighted = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
# NOTE: cv="prefit" was removed in sklearn 1.8 — wrap the fitted model in FrozenEstimator.
recalib = CalibratedClassifierCV(FrozenEstimator(weighted), method="isotonic").fit(Xcal, ycal)

print(f"{'model':<34}{'ROC-AUC':>9}{'Brier':>9}{'ECE':>8}{'mean pred':>11}")
print("-" * 71)
print(f"{'true positive rate':<34}{'—':>9}{'—':>9}{'—':>8}{yho.mean():>11.4f}")
for name, m in [("logistic (unweighted)", base),
                ("logistic (class_weight=balanced)", weighted),
                ("  + isotonic recalibration", recalib)]:
    p = m.predict_proba(Xho)[:, 1]
    print(f"{name:<34}{roc_auc_score(yho, p):>9.3f}{brier_score_loss(yho, p):>9.4f}"
          f"{ece(yho, p):>8.4f}{p.mean():>11.4f}")

print()
print("Look at ROC-AUC: essentially identical across all three. Ranking is untouched.")
print("Look at 'mean pred' against the true positive rate: the balanced model predicts")
print("a positive rate roughly an order of magnitude too high. It ranks just as well and")
print("its probabilities are unusable for any expected-value calculation.")
print()
print("This is the trap: class_weight='balanced' looks like a free improvement because")
print("recall goes up and AUC doesn't move. It silently breaks every downstream")
print("computation that treats the output as a probability.")

## 5. Class Imbalance

| Approach | Mechanism | Watch out for |
|---|---|---|
| **Do nothing, move the threshold** | Train normally, tune the operating point | Usually the right first move — and it preserves calibration |
| **Class weights** | Penalize minority errors more in the loss | Breaks calibration (above) |
| **Undersample majority** | Discard majority examples | Throws away real data; increases variance |
| **Oversample minority** | Duplicate minority examples | Overfits duplicates; must happen *inside* CV folds |
| **SMOTE** | Synthesize interpolated minority points | Interpolating in high dimensions can create implausible examples; weaker than its reputation |
| **Collect more minority data** | The actual fix | Slow, often impossible, always best |
| **Reframe as anomaly detection** | Model the majority, flag deviations | Appropriate at extreme imbalance (<0.1%) |

**Start by doing nothing.** Train the model, evaluate with PR-AUC, and tune the threshold
against costs. A large fraction of "imbalance problems" are really "we reported accuracy and
used a 0.5 threshold" problems, and resampling gets applied as a fix for something that was
never broken.

**The leakage trap:** resampling must happen *inside* each cross-validation fold. Oversample
before splitting and copies of the same minority row land in both train and validation, so your
validation score measures memorization. This is a very common interview probe.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Leakage demo: oversample before vs inside the CV split.
def naive_oversample(X, y, seed=0):
    r = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]
    need = (y == 0).sum() - len(pos)
    extra = r.choice(pos, size=need, replace=True)
    idx = np.concatenate([np.arange(len(y)), extra])
    return X[idx], y[idx]

Xs, ys = X[:6000], y[:6000]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
clf = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=2000))])

# WRONG: oversample the whole dataset, then cross-validate over it.
Xo, yo = naive_oversample(Xs, ys)
wrong = cross_val_score(clf, Xo, yo, cv=cv, scoring="average_precision").mean()

# RIGHT: cross-validate on the original data; the model sees the true distribution.
right = cross_val_score(clf, Xs, ys, cv=cv, scoring="average_precision").mean()

print(f"{'protocol':<44}{'PR-AUC':>9}")
print("-" * 53)
print(f"{'oversample BEFORE split (leaks)':<44}{wrong:>9.3f}")
print(f"{'no resampling, honest CV':<44}{right:>9.3f}")
print(f"\nInflation from leakage: {wrong - right:+.3f}")
print()
print("Duplicated minority rows land in both train and validation folds, so the model")
print("is scored partly on rows it memorized. The optimistic number ships, production")
print("underperforms, and the postmortem blames drift.")

## 6. Connecting Offline Metrics to Online Ones

This is the part that most distinguishes an MLE from an analyst, and it's where an
experimentation background is a real asset.

```
Business metric        revenue, retention, fraud loss        weeks to move, high noise
      ^
Online proxy           CTR, completion rate, block rate      days, A/B testable
      ^
Offline metric         PR-AUC, NDCG@10, calibration error    minutes, every commit
```

You optimize the bottom because it's fast. You are *paid* for the top. **The link between them
is an assumption, and it needs to be validated, not assumed.**

### How the link breaks
- **Ranking metrics ignore position bias.** Offline NDCG computed on logged data rewards
  reproducing what was already shown.
- **Offline evaluation is counterfactual-blind.** You only observe outcomes for items that were
  actually served — the classic recsys problem.
- **The offline metric may not be the constraint.** Improving PR-AUC when the review team is
  capacity-bound at 500 cases/day changes nothing you can see.
- **Guardrails move instead.** Precision improves, recall drops, and complaint volume rises.

### What to do
1. Pick an offline metric and **verify empirically that it correlates with your online metric**
   across several past experiments. If it doesn't, it's the wrong offline metric.
2. Always ship with **guardrail metrics** — the things that must not get worse.
3. Expect the offline gain to shrink online. A large offline win that produces no online
   movement is the normal outcome, not an anomaly.

> 💡 **Interview Tip:** If you have an experimentation background, this is where to spend your
> credibility. "I'd validate that the offline metric actually predicts the online one before
> trusting it as a decision gate" is a sentence most candidates don't say, and it's exactly the
> instinct an MLE team wants.

## What Changes in Production

- **The threshold is a deployed artifact**, versioned alongside the model. It needs an owner, a
  change process, and monitoring — and it will need re-tuning when the base rate drifts.
- **Calibration decays** as the population shifts. Monitor predicted-positive-rate against
  actual-positive-rate over time; divergence is an early warning that fires before your
  accuracy metric does.
- **Labels arrive late, or never.** Fraud confirms in weeks; churn in months. Your online metric
  is delayed, so you need leading indicators (score distribution, flag rate, feature drift) to
  detect problems before ground truth exists.
- **The score distribution is your most sensitive monitor.** It requires no labels, computes in
  real time, and shifts before performance does. Alert on it.

## Common Interview Questions

**Q: Your model has 99% accuracy. Are you happy?**
Not without knowing the base rate. At a 1% positive rate, predicting the majority class always
gives 99% and catches nothing. I'd want the confusion matrix, PR-AUC against its baseline (the
positive rate), and the precision and recall at the operating threshold we'd actually deploy.

**Q: ROC-AUC or PR-AUC?**
PR-AUC when positives are rare and you act on individual predictions, because FPR has TN in the
denominator — so with vast numbers of negatives, thousands of false positives barely move the
ROC curve while precision collapses. ROC-AUC is fine for balanced problems and comparing
rankers. And always report PR-AUC's baseline, since it equals the positive rate and the number
is meaningless without it.

**Q: How do you choose a classification threshold?**
By asking what the errors cost. I'd get the relative cost of a false positive and a false
negative from whoever owns the outcome, sweep thresholds to minimize expected cost, and check
whether a capacity constraint binds first — if the review team handles 500 cases a day, that
sets the threshold regardless of what the cost curve says. And I'd re-tune when the base rate
shifts.

**Q: A model has AUC 0.95 but its probabilities are useless. How?**
AUC measures ranking only; it's invariant to any monotonic transform of the scores. Squaring
every prediction leaves AUC identical and destroys calibration. If the output feeds an
expected-value computation — a bid, an expected loss, an escalation rule with a stated
confidence — you need calibration measured separately, via reliability curves, Brier score, or
ECE, and fixed with Platt or isotonic scaling on held-out data.

**Q: What's wrong with `class_weight='balanced'`?**
Nothing, if you only need ranking. But it changes the effective base rate the model trains
against, so its outputs no longer estimate true probabilities — typically over-predicting the
positive class substantially. AUC is unchanged, which makes it look free, so it silently breaks
every downstream calculation that treats the score as a probability. Recalibrate afterward on
held-out data, or skip the weighting and move the threshold instead.

**Q: Where does resampling go relative to cross-validation?**
Inside each fold, always. Oversampling before splitting puts duplicates of the same minority
row into both train and validation, so validation partly measures memorization and the score is
inflated. Use a pipeline so the resampling step is fitted per fold.

**Q: Offline metrics improved but the A/B test was flat. What happened?**
Several possibilities and I'd check them in order. The offline metric may not predict the
online one — worth validating against past experiments before trusting it as a gate. Offline
evaluation on logged data is counterfactual-blind and biased toward reproducing what was
already shown. The metric may not be the binding constraint. Or a guardrail moved and offset
the gain. The general lesson is that offline improvement is a hypothesis about online
improvement, not evidence of it.

## Key Takeaways
- Accuracy is uninformative under imbalance; always report the base rate alongside any metric
- PR-AUC reflects rare-positive performance honestly because it has no TN term; its baseline is the positive rate
- The threshold is a business decision independent of the model — often a bigger lever than retraining
- Ask what errors cost, and check whether a capacity constraint binds before optimizing anything
- Calibration ≠ ranking: AUC is invariant to monotonic transforms that destroy probability quality
- Calibration matters whenever the score is multiplied by something or shown to a human
- `class_weight='balanced'` and resampling both break calibration while leaving AUC unchanged
- Try "do nothing and tune the threshold" before any resampling scheme
- Resample inside CV folds, never before splitting
- Validate that your offline metric actually predicts the online one — that link is an assumption